In [1]:
!pip install sacrebleu
!pip install transformers
!pip install peft
!pip install trl
!pip install accelerate
!pip install evaluate

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 159.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 162.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 753.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 226.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 257.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 32.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] 

In [3]:
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, TaskType
from trl import SFTTrainer
from datasets import load_dataset
model_name = "Qwen/Qwen3-0.6B"
output = os.path.join(direc,"qwen3_lora")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
#set chatML template
tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)
peft_config = LoraConfig(
    r = 64,
    lora_alpha=128,
    lora_dropout=0.05,
    #fine tuning all layers according to JUG
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type=TaskType.CAUSAL_LM,
    bias="none"
)
dataset =load_dataset("json",data_files=train_file, split="train")
training_args =TrainingArguments(
    output_dir= output,
    per_device_train_batch_size = 2, 
    gradient_accumulation_steps = 32,
    # 1e-4 suggested by JUG
    learning_rate = 1e-4,
    num_train_epochs = 2,
    # linear learning rate scheduler suggested by JUG
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    logging_steps=100,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    group_by_length=True
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(output)
tokenizer.save_pretrained(output)

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
100,2.812000
200,1.681800
300,1.419700
400,1.278500
500,1.193100
600,1.131300
700,1.095700
800,1.066900
900,1.035600
1000,1.024400


('/dss/dsshome1/09/go34jos2/qwen3_lora/tokenizer_config.json',
 '/dss/dsshome1/09/go34jos2/qwen3_lora/special_tokens_map.json',
 '/dss/dsshome1/09/go34jos2/qwen3_lora/chat_template.jinja',
 '/dss/dsshome1/09/go34jos2/qwen3_lora/vocab.json',
 '/dss/dsshome1/09/go34jos2/qwen3_lora/merges.txt',
 '/dss/dsshome1/09/go34jos2/qwen3_lora/added_tokens.json',
 '/dss/dsshome1/09/go34jos2/qwen3_lora/tokenizer.json')

In [9]:
import re
import evaluate
from peft import PeftModel
from tqdm import tqdm
val_src =os.path.join(direc,"dev.de-hsb.de")
val_tgt = os.path.join(direc, "dev.de-hsb.hsb")
#extract hsb
def extract_translation(text):
    match = re.search(r"<hsb>\s*(.*?)\s*</hsb>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    dtype=torch.bfloat16,  
    device_map="auto",
    attn_implementation="eager"
)
model = PeftModel.from_pretrained(base_model,output)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(output)
tokenizer.padding_side = "left"
if tokenizer.chat_template is None:
     tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
with open(val_src, encoding='utf-8') as f:
    val_s = [line.strip() for line in f.readlines() if line.strip()]
with open(val_tgt, encoding='utf-8') as f:
    val_t = [line.strip() for line in f.readlines() if line.strip()]
instruction_text = "Translate the following German text to Upper Sorbian.\nPut it in this format <hsb> Upper Sorbian translation </hsb>."
predictions = []
batch_size = 16
for i in tqdm(range(0, len(val_s),batch_size)):
    batch_src = val_s[i:i+batch_size]
    batch_prompts = []
    for src in batch_src:
        messages = [
            {"role": "system", "content": instruction_text},
            {"role": "user", "content": f"<deu> {src} </deu>"}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch_prompts.append(text)
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )
    generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    raw_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    clean_preds = [extract_translation(p) for p in raw_preds]
    predictions.extend(clean_preds)
        
        

100%|██████████| 250/250 [30:55<00:00,  7.42s/it]


In [10]:
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
bleu_result = sacrebleu.compute(predictions=predictions, references=[[r] for r in val_t])
chrf_result = chrf.compute(predictions=predictions, references=[[r] for r in val_t], word_order=2)
print(f"SacreBLEU: {bleu_result['score']:.2f}")
print(f"chrF++:    {chrf_result['score']:.2f}")

SacreBLEU: 50.63
chrF++:    72.00
